In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.preprocessing import StandardScaler

In [ ]:
# Carregar e explorar os dados
df = pd.read_csv('species_dataset.csv')
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df[['diet_type', 'skin_type', 'social_behavior']].describe()

In [ ]:
# Preparar os dados
X = pd.get_dummies(df.drop(columns=['species_id']), columns=['diet_type', 'skin_type', 'social_behavior'])
X = StandardScaler().fit_transform(X)

In [ ]:
# Clusterização hierárquica
Z = linkage(X, method='ward')

In [ ]:
# Dendrograma
plt.figure(figsize=(15, 6))
dendrogram(Z, truncate_mode='lastp', p=30)
plt.show()

In [ ]:
# Corte em 10 clusters
df['cluster'] = fcluster(Z, t=10, criterion='maxclust')
df['cluster'].value_counts().sort_index()

In [ ]:
# Descrição dos grupos
df.groupby('cluster').mean(numeric_only=True)

In [ ]:
df.groupby('cluster')[['diet_type', 'skin_type', 'social_behavior']].agg(lambda x: x.mode()[0])

## Análise dos resultados

O dendrograma foi cortado em 10 clusters, com tamanhos entre 36 e 127 espécies.

As variáveis numéricas (massa, cauda, expectativa de vida, olhos) ficaram com médias muito parecidas entre os grupos, então quase não ajudam a separar as espécies. Quem define os grupos são principalmente as variáveis categóricas e o veneno:

- Cobertura do corpo é o traço mais forte: há grupos de penas (1, 2 e 3), de pelos (4 e 5), de pele (6, 7 e 9) e de escamas (8 e 10).
- Dentro de cada cobertura, os grupos se dividem por dieta e comportamento social. Por exemplo, entre os de penas, o 1 é onívoro e vive em pares, o 2 é carnívoro e solitário, e o 3 é herbívoro e vive em pares.
- O cluster 5 é o único em que todas as espécies têm veneno, o que faz dele o grupo mais distinto.
- O cluster 10 tem a maior média de patas (4,7) e mais espécies venenosas que a média, o que o aproxima de um grupo tipo artrópode.

Pensando como uma árvore filogenética simulada, a primeira grande divisão seria pela cobertura do corpo, como se fosse uma separação antiga entre linhagens, e as divisões mais recentes seriam pela dieta e pelo comportamento social. O veneno aparece como um traço que surge em um ramo específico (cluster 5).

Como os dados são sintéticos e as variáveis numéricas têm pouca variação entre grupos, os agrupamentos dependem muito da codificação das categóricas. Ou seja, o resultado mostra mais combinações de categorias do que relações evolutivas reais.